# A. gamma 0.995 PPO

보상은 베이스라인 그대로이고 gamma만 0.995다. 환경 코드는 바꾸지 않는다.

Kaggle에서 Internet을 켠다. Accelerator는 CPU로 둔다. SUMO가 CPU라 GPU를 켜도 학습 시간이 줄지 않는다.
저장소가 비공개면 Add-ons → Secrets에 `GITHUB_TOKEN`을 넣는다. 토큰은 노트북 안에 적지 않는다.

학습이 12시간을 넘기면 출력의 `model.resume.pt`를 받아 다음 세션에서 resume 칸을 실행한다.

In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working"
SUMO_SRC = f"{ROOT}/sumo-1.27.1"
SUMO_PREFIX = f"{ROOT}/sumo-install"
os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f"{SUMO_PREFIX}/bin/sumo"):
    subprocess.check_call("apt-get update && apt-get install -y cmake g++ libxerces-c-dev libgdal-dev libproj-dev curl", shell=True)
    archive = f"{ROOT}/sumo-src-1.27.1.tar.gz"
    subprocess.check_call(["curl", "-L", "-o", archive, "https://sumo.dlr.de/releases/1.27.1/sumo-src-1.27.1.tar.gz"])
    subprocess.check_call(["tar", "-xzf", archive, "-C", ROOT])
    build = f"{SUMO_SRC}/build"
    os.makedirs(build, exist_ok=True)
    subprocess.check_call(["cmake", "-S", SUMO_SRC, "-B", build, "-DCMAKE_BUILD_TYPE=Release", f"-DCMAKE_INSTALL_PREFIX={SUMO_PREFIX}"])
    subprocess.check_call(["cmake", "--build", build, "-j2"])
    subprocess.check_call(["cmake", "--install", build])
os.environ["SUMO_HOME"] = SUMO_PREFIX
os.environ["PATH"] = f"{SUMO_PREFIX}/bin:" + os.environ.get("PATH", "")
print(subprocess.check_output(["sumo", "--version"], text=True).splitlines()[0])


In [ ]:
import os, subprocess
COMMIT = "5b4b81bb81f00807a68d0b22b2de3c0d11c134f6"
REPO = "/kaggle/working/repo-gamma"
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("GITHUB_TOKEN 없음. 공개 저장소가 아니면 Secrets에 넣는다:", exc)
url = "https://github.com/TranserZ/Artificial-Intelligence-and-Control-for-Autonomous-Driving.git"
if token:
    url = url.replace("https://", f"https://{token}@")
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "--branch", "model/exp-ppo-gamma", url, REPO])
subprocess.check_call(["git", "checkout", COMMIT], cwd=REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True))


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "gymnasium==1.4.0", "matplotlib==3.11.2", "numpy==2.5.4",
    "tensorboard==2.21.0", "traci==1.28.0"])
import torch, traci
print("torch", torch.__version__, "traci", traci.__version__)


In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo-gamma"
OUT = "/kaggle/working/exp-ppo-gamma"
os.makedirs(OUT, exist_ok=True)
env = os.environ.copy()
cmd = [sys.executable, "train.py",
       "--experiment", "models/exp-ppo-gamma/experiment.json",
       "--seed", "0", "--replay-prob", "0", "--eval-interval", "0",
       "--total-steps", "300000", "--save-steps", "300000",
       "--out-dir", OUT]
resume = os.path.join(OUT, "model.resume.pt")
if os.path.exists(resume):
    cmd += ["--resume", resume]
    print("이어서 학습", resume)
subprocess.check_call(cmd, cwd=REPO, env=env)


In [ ]:
import os, subprocess
OUT = "/kaggle/working/exp-ppo-gamma"
archive = "/kaggle/working/exp-ppo-gamma-300k.tar.gz"
subprocess.check_call(["tar", "-C", "/kaggle/working", "-czf", archive, "exp-ppo-gamma"])
print(archive, os.path.getsize(archive))
for name in ("config.json", "training_log.csv", "final.pt", "final.resume.pt", "step300k.pt", "model.resume.pt"):
    path = os.path.join(OUT, name)
    print(name, "있음" if os.path.exists(path) else "없음")
